# Distance to Nearest Pharmacy from SAL Centroids
## Euclidean and OSMnx Network Distance (Pedestrian + Drive)

**Tess Vu**

This notebook computes the distance from each Small Area Layer (SAL) centroid
to the nearest geocoded pharmacy using three methods:

1. Euclidean (straight-line) distance via scipy KDTree.
2. Pedestrian network distance via `OSMnx` walk graph.
3. Drive network distance via `OSMnx` drive graph.

Network graphs are saved as `.graphml` files for reuse.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
from scipy.spatial import cKDTree
from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib as mpl
import mapclassify
import warnings
import os
import time

warnings.filterwarnings("ignore")
print(f"OSMnx Version: {ox.__version__}")

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()          # repository root; notebooks run from notebooks/
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERMEDIATE = DATA / "intermediate"
FINAL = DATA / "final"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import dair_style as ds

IN_SAL_DEDUP = INTERMEDIATE / "sal_w_ward_dedup" / "sal_w_ward_dedup.shp"
IN_POP_PRED = INTERMEDIATE / "pop_pred_final.csv"
IN_PHARMACIES = RAW / "PHARMACIES_MASTER_FINAL.csv"
OUT_NETWORK_DIR = INTERMEDIATE / "networks"


## CONFIGURATION
Update file paths below to match local directory structure.
All paths are relative to the notebook location, following the
partner notebook conventions.

In [ ]:
# FILE PATHS
# All input and output paths are defined in the paths cell above.
# Directory for saved network graphs and results.
os.makedirs(OUT_NETWORK_DIR, exist_ok = True)

# Projected CRS for distance calculations in meters.
# South African standard: Hartebeesthoek94 / Lo29 for central SA.
PROJECTED_CRS = "EPSG:32735"

# Province names for filtering and labeling.
PROVINCES = ["Gauteng", "KwaZulu-Natal"]

print(f"Output Directory: {os.path.abspath(OUT_NETWORK_DIR)}")


## LOAD SAL GEOMETRIES AND POPULATION ESTIMATES
Load the SAL shapefile that has ward assignments, then join
with the population estimates produced by the dasymetric notebook.

In [ ]:
# Load SAL shapefile with geometry and ward assignments.
sal_geo = gpd.read_file(IN_SAL_DEDUP)
print(f"SAL shapefile loaded: {sal_geo.shape[0]} features")
print(f"CRS: {sal_geo.crs}")
print(f"Columns: {list(sal_geo.columns)}")

In [ ]:
# Load population estimates from the dasymetric step-down.
pop_est = pd.read_csv(IN_POP_PRED)
print(f"Population estimates loaded: {pop_est.shape[0]} rows")
print(f"Columns: {list(pop_est.columns)}")

In [ ]:
# Join population estimates onto SAL geometries.
# The join key is EA_CODE (SAL identifier).
sal = sal_geo.merge(
    pop_est[["EA_CODE", "sal2023_est", "WardID"]],
    on = "EA_CODE",
    how = "left"
)

# Report join quality.
matched = sal["sal2023_est"].notna().sum()
total = sal.shape[0]
print(f"JOIN RESULTS: {matched}/{total} SALs matched ({matched/total*100:.1f}%)")

# Drop SALs with no population estimate.
sal = sal[sal["sal2023_est"].notna()].copy()
print(f"SALs retained after dropping unmatched: {sal.shape[0]}")

## COMPUTE SAL CENTROIDS
Project to a metric CRS, compute geometric centroids, then extract
coordinates for distance calculations.

**Limitation:** Geometric centroids may fall in uninhabited areas
(rivers, parks, industrial zones) for irregularly shaped SALs.
Population-weighted centroids using building footprints would be
more defensible but require additional data processing.

In [ ]:
# Project SAL geometries to metric CRS for distance calculations.
sal_proj = sal.to_crs(PROJECTED_CRS)

# Compute geometric centroids in projected space.
sal_proj["centroid_geom"] = sal_proj.geometry.centroid

# Extract centroid coordinates in projected CRS (meters).
sal_proj["centroid_x"] = sal_proj["centroid_geom"].x
sal_proj["centroid_y"] = sal_proj["centroid_geom"].y

# Also compute centroids in WGS84 for OSMnx (which expects lat/lng).
sal_wgs84 = sal.to_crs("EPSG:4326")
sal_wgs84["centroid_wgs84"] = sal_wgs84.geometry.centroid
sal_proj["centroid_lat"] = sal_wgs84["centroid_wgs84"].y.values
sal_proj["centroid_lng"] = sal_wgs84["centroid_wgs84"].x.values

print(f"SAL CENTROIDS COMPUTED: {sal_proj.shape[0]} centroids")
print(f"Projected CRS: {PROJECTED_CRS}")
print(f"Centroid X range: {sal_proj['centroid_x'].min():.0f} to {sal_proj['centroid_x'].max():.0f} m")
print(f"Centroid Y range: {sal_proj['centroid_y'].min():.0f} to {sal_proj['centroid_y'].max():.0f} m")

## LOAD GEOCODED PHARMACIES
Load the fully geocoded pharmacy file and convert to a GeoDataFrame.

In [ ]:
# Load geocoded pharmacies.
pharm_df = pd.read_csv(IN_PHARMACIES)
print(f"Pharmacies loaded: {pharm_df.shape[0]} rows")
print(f"Columns: {list(pharm_df.columns)}")

# Check for lat/lng columns and filter valid coordinates.
assert "LAT" in pharm_df.columns, "Missing 'LAT' column in pharmacy file."
assert "LNG" in pharm_df.columns, "Missing 'LNG' column in pharmacy file."

# Drop rows with missing coordinates.
valid_coords = pharm_df["LAT"].notna() & pharm_df["LNG"].notna()
print(f"Pharmacies with valid coordinates: {valid_coords.sum()}/{pharm_df.shape[0]}")
pharm_df = pharm_df[valid_coords].copy()

# Convert to GeoDataFrame.
pharm_gdf = gpd.GeoDataFrame(
    pharm_df,
    geometry = gpd.points_from_xy(pharm_df["LNG"], pharm_df["LAT"]),
    crs = "EPSG:4326"
)

# Project to metric CRS.
pharm_proj = pharm_gdf.to_crs(PROJECTED_CRS)
pharm_proj["pharm_x"] = pharm_proj.geometry.x
pharm_proj["pharm_y"] = pharm_proj.geometry.y

print(f"PHARMACIES READY: {pharm_proj.shape[0]} geocoded locations")

In [ ]:
# Quick sanity check: bounding box of pharmacies vs SALs.
print("BOUNDING BOX COMPARISON (WGS84)")
sal_bounds = sal.to_crs("EPSG:4326").total_bounds
pharm_bounds = pharm_gdf.total_bounds
print(f"SALs: W={sal_bounds[0]:.3f}, S={sal_bounds[1]:.3f}, E={sal_bounds[2]:.3f}, N={sal_bounds[3]:.3f}")
print(f"Pharmacies: W={pharm_bounds[0]:.3f}, S={pharm_bounds[1]:.3f}, E={pharm_bounds[2]:.3f}, N={pharm_bounds[3]:.3f}")

## EUCLIDEAN DISTANCE TO NEAREST PHARMACY
Use a KDTree for efficient nearest-neighbor lookup in projected
coordinates (meters). This gives straight-line distances that
underestimate true travel distances but serve as a useful lower bound.

### NOTE: Notebook `sal_pharmacy_distance_k3` supersedes this notebook's distance outputs, this is just a check.

In [ ]:
# Build KDTree from pharmacy locations in projected CRS.
pharm_coords = np.column_stack([pharm_proj["pharm_x"].values, pharm_proj["pharm_y"].values])
tree = cKDTree(pharm_coords)

# Query nearest pharmacy for each SAL centroid.
sal_coords = np.column_stack([sal_proj["centroid_x"].values, sal_proj["centroid_y"].values])
distances_m, indices = tree.query(sal_coords, k = 1)

# Store results.
sal_proj["euclidean_dist_m"] = distances_m
sal_proj["euclidean_dist_km"] = distances_m / 1000.0
sal_proj["nearest_pharm_idx"] = indices

print("EUCLIDEAN DISTANCE COMPUTED")
print(sal_proj["euclidean_dist_km"].describe().to_string())

## EUCLIDEAN DISTANCE HEATMAP TEST
Choropleth map of SAL polygons colored by Euclidean distance to the nearest pharmacy, shown per province.

Notebook `sal_pharmacy_distance_k3.ipynb` will formally analyze the distance outputs and produce summary statistics and visualizations.

In [ ]:
# Prepare data for mapping (keep original geometry, not centroids).
sal_map_data = sal_proj.copy()
sal_map_data = sal_map_data.set_geometry("geometry")

# Determine province column name from shapefile.
# Common variants: PR_NAME, PROVINCE, Province.
prov_col = None
for candidate in ["PR_NAME", "PROVINCE", "Province", "province"]:
    if candidate in sal_map_data.columns:
        prov_col = candidate
        break

if prov_col is None:
    print("WARNING: No province column found. Plotting all SALs together.")
    prov_col = "ALL"
    sal_map_data["ALL"] = "All Provinces"

print(f"Province column: {prov_col}")
print(f"Unique values: {sal_map_data[prov_col].unique()}")

In [ ]:
# Define distance bins in kilometers.
dist_bins_km = [0, 1, 2, 3, 5, 10, 15, 25, 50]

fig, axes = plt.subplots(1, 2, figsize = (18, 10))
cmap = ds.DISTANCE_CMAP

provinces_in_data = sal_map_data[prov_col].unique()

for i, prov in enumerate(provinces_in_data[:2]):
    subset = sal_map_data[sal_map_data[prov_col] == prov].copy()
    subset = subset[subset["euclidean_dist_km"].notna()]

    subset.plot(
        column = "euclidean_dist_km",
        cmap = cmap,
        scheme = "UserDefined",
        classification_kwds = {"bins": dist_bins_km},
        legend = True,
        legend_kwds = {"title": "Distance (km)", "loc": "lower right", "fontsize": 8},
        ax = axes[i],
        edgecolor = "none",
        linewidth = 0
    )

    axes[i].set_title(f"{prov}: Euclidean Distance to Nearest Pharmacy", fontsize = 12)
    axes[i].axis("off")

plt.suptitle("SAL Centroid Euclidean Distance to Nearest Pharmacy", fontsize = 14, y = 1.02)
plt.tight_layout()

#plt.savefig(output_path, dpi = 300, bbox_inches = "tight")
plt.show()
#print(f"Map saved to {output_path}")

## DOWNLOAD OSMnx NETWORK GRAPHS
Download pedestrian (walk) and drive networks for each province.
These are large downloads; each graph is saved as `.graphml`
for reuse without re-downloading.

**Note:** This step may take 10-30+ minutes per province depending
on network size and connection speed. The saved `.graphml` files
can be reloaded in future sessions.

In [ ]:
# Configure OSMnx settings for large downloads.
ox.settings.use_cache = True
ox.settings.log_console = True
ox.settings.timeout = 600

# Define place queries for each province.
place_queries = {
    "Gauteng": "Gauteng, South Africa",
    "KwaZulu-Natal": "KwaZulu-Natal, South Africa"
}

# Network types to download.
network_types = {
    "walk": "pedestrian",
    "drive": "drive"
}

# Use cache because files are large.
print(f"Cache Enabled: {ox.settings.use_cache}")
print(f"Timeout: {ox.settings.timeout}s")

In [ ]:
# Download and save network graphs.
# Store graph references for distance computation.
graphs = {}

for prov_name, place_query in place_queries.items():
    for net_type, net_label in network_types.items():
        graph_key = f"{prov_name}_{net_type}"
        graphml_path = OUT_NETWORK_DIR / f"network_{prov_name.lower().replace('-', '_')}_{net_type}.graphml"

        # Check if graph already exists on disk.
        if os.path.exists(graphml_path):
            print(f"CACHED GRAPH: {graph_key}")
            G = ox.load_graphml(graphml_path)
        else:
            print(f"DOWNLOADING {net_label.upper()} NETWORK: {place_query}")
            start_time = time.time()
            G = ox.graph_from_place(place_query, network_type = net_type)
            elapsed = time.time() - start_time
            print(f"Download complete in {elapsed:.1f} seconds.")

            # Add edge lengths if not already present.
            if not all("length" in data for _, _, data in G.edges(data = True)):
                G = ox.distance.add_edge_lengths(G)

            # Save to disk.
            ox.save_graphml(G, graphml_path)
            print(f"Graph saved to {graphml_path}")

        # Store reference.
        graphs[graph_key] = G
        print(f"  Nodes: {G.number_of_nodes():,}, Edges: {G.number_of_edges():,}\n")

print(f"Graph Keys: {list(graphs.keys())}")